In [10]:
# Required imports for Selenium automation and clipboard handling
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import pyperclip
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# Chrome Options: Allow clipboard access and (optionally) headless operation
options = Options()
# options.add_argument("--headless")  # Uncomment for headless mode
options.add_experimental_option("prefs", {
    "profile.default_content_setting_values.clipboard": 1,
    "profile.content_settings.exceptions.clipboard": {
        "[*.]aion-archives.net,*": {
            "setting": 1
        }
    }
})

# ─────────────────────────────────────────────────────────────────────────────
# Functions for interacting with the AION Archives site

# Click the "Open Connection" button to start the session
def open_connection():
    enter_btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.ID, "enterButton"))
    )
    enter_btn.click()
    print("Connection opened.")

# Click a single digit button on the calculator UI
def click_digit(digit):
    selector = f'button[data-digit="{digit}"]'
    btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
    driver.execute_script("arguments[0].removeAttribute('disabled')", btn)
    driver.execute_script("arguments[0].scrollIntoView(true);", btn)
    wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
    btn.click()
    sleep(0.3)  # Slight delay between digit presses

# Click multiple digits in sequence (used for frequency entry)
def click_digits(code):
    for digit in code:
        click_digit(digit)

# Paste a FEN string into the input by copying it to clipboard and clicking the help icon
def paste_fen(fen):
    pyperclip.copy(fen)  # Set clipboard content
    try:
        help_btn = wait.until(EC.element_to_be_clickable((By.ID, "helpButton")))
        driver.execute_script("arguments[0].scrollIntoView(true);", help_btn)
        help_btn.click()  # Trigger paste from clipboard
        print(f"✅ Pasted FEN: {fen}")
    except TimeoutException:
        print("⚠️ Paste button never became clickable. Frequency might be invalid.")

# Click the Submit button
def submit():
    submit_btn = wait.until(EC.element_to_be_clickable((By.ID, "submitButton")))
    driver.execute_script("arguments[0].scrollIntoView(true);", submit_btn)
    submit_btn.click()

# Retrieve the most recent alert or error message shown on the page
def get_alert_message():
    try:
        # Check for latest alert
        alert_boxes = driver.find_elements(By.CLASS_NAME, "alert-message")
        if alert_boxes:
            return alert_boxes[-1].text.strip()
    except:
        pass
    try:
        # Fallback: check for latest error
        error_box = driver.find_element(By.CLASS_NAME, "error-message")
        return error_box.text.strip()
    except:
        return "⚠️ No alert or error message found"

# Clear all previous alerts and errors before submitting new inputs
def clear_output():
    try:
        alert_boxes = driver.find_elements(By.CLASS_NAME, "alert-message")
        for alert in alert_boxes:
            driver.execute_script("arguments[0].remove();", alert)

        error_boxes = driver.find_elements(By.CLASS_NAME, "error-message")
        for error in error_boxes:
            driver.execute_script("arguments[0].remove();", error)

        print("Previous alerts and errors cleared.")
    except Exception as e:
        print(f"⚠️ Error clearing messages: {str(e)}")

# ─────────────────────────────────────────────────────────────────────────────
# Main Execution

# Load TSV data into DataFrame (ensure file is in the same directory or use full path)
df = pd.read_csv("data.tsv", sep="\t")

# Start browser session
driver = webdriver.Chrome(options=options)
driver.get("https://www.aion-archives.net/")
wait = WebDriverWait(driver, 10)

# Open connection to begin input
open_connection()
sleep(10)

# Loop through first few rows (adjust .head(N) as needed)
results = []
for i, row in df.head(5).iterrows():
    try:
        click_digits(str(row['frequency']).zfill(4))  # Pad frequency like '0042'
        submit()
        sleep(5)

        paste_fen(row['fen'])
        submit()
        sleep(10)

        alert = get_alert_message()
        results.append({
            "frequency": row['frequency'],
            "fen": row['fen'],
            "alert_message": alert,
            "success": alert == "ALERT: Data Found!"
        })

        print(f"{row['frequency']} → {alert}")

    except Exception as e:
        # Capture any exceptions and mark the row as failed
        results.append({
            "frequency": row['frequency'],
            "fen": row['fen'],
            "alert_message": str(e),
            "success": False
        })

    sleep(5)  # Wait before next loop iteration
    clear_output()  # Ensure output area is clean for next run

# Save results to CSV
final_df = pd.DataFrame(results)
final_df.to_csv("aion_results.csv", index=False)
print("Done.")

# Close browser
driver.quit()


Connection opened.
✅ Pasted FEN: kBrPQbRq/r6p/P6R/q2nn2B/B2nn2q/R6k/p6r/qPnbKpRr
1849 → ALERT: Data Found!
Previous alerts and errors cleared.
✅ Pasted FEN: nqKRnQkr/p6r/q6N/B2PP2Q/r2PP2k/P6R/K6n/pqNrkQnR
2979 → ALERT: Data Found!
Previous alerts and errors cleared.
✅ Pasted FEN: kbKPrBkK/q6K/B6P/R6N/K6b/K6q/b6p/KnBqpNbr
646 → ERROR: Quantum spin Fai0x71=91207
Previous alerts and errors cleared.
✅ Pasted FEN: bqBRpQbR/Q6R/b6R/R2RR2R/p2RR2R/q6R/N6R/PpRBqPrR
61 → ALERT: Data Found!
Previous alerts and errors cleared.
✅ Pasted FEN: rKRNqkrn/b6p/N6k/p6b/k6N/B6P/n6Q/rkrnQKRp
1266 → ALERT: Data Found!
Previous alerts and errors cleared.
Done.
